# Streamlit Attention Heatmap App — HuggingFace Integration

**Repo:** [ai-attention-token-viz](https://github.com/TylrDn/ai-attention-token-viz)  
**Phase:** 3 — Notebook Pipelines

This notebook:
1. Defines a reusable attention-extraction helper compatible with any HF causal/encoder model.
2. Renders an interactive heatmap with Plotly.
3. Generates the `streamlit_app.py` app file used by the sister repo.

### References
- BERT: Devlin et al. (2018) https://arxiv.org/abs/1810.04805
- BERTviz: Vig (2019) https://arxiv.org/abs/1906.05714
- HuggingFace Transformers: Wolf et al. (2020) https://arxiv.org/abs/1910.03771

In [ ]:
from __future__ import annotations

import logging
import os

import torch
import wandb

logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
logger = logging.getLogger(__name__)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WANDB_DISABLED = os.getenv("WANDB_DISABLED", "false").lower() == "true"
DEFAULT_MODEL = os.getenv("VIZ_MODEL", "bert-base-uncased")
DEFAULT_TEXT = os.getenv("VIZ_TEXT", "The transformer architecture changed natural language processing.")

logger.info("Device: %s  |  Model: %s", DEVICE, DEFAULT_MODEL)

In [ ]:
from transformers import AutoModel, AutoTokenizer


def extract_attention(
    model_id: str,
    text: str,
    device: torch.device = DEVICE,
) -> tuple[list[str], list[torch.Tensor]]:
    """Extract per-layer, per-head attention weights for a given text.

    Returns (tokens, attention_layers) where attention_layers[i] has
    shape (num_heads, seq_len, seq_len).

    Compatible with any HuggingFace model that outputs attentions.
    Devlin et al. (2018) https://arxiv.org/abs/1810.04805
    """
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id, output_attentions=True)
    model = model.to(device).eval()

    enc = tokenizer(text, return_tensors="pt").to(device)
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])

    with torch.no_grad():
        out = model(**enc)

    # out.attentions: tuple of (batch=1, heads, seq, seq) per layer
    attention_layers = [layer[0].cpu() for layer in out.attentions]
    logger.info(
        "Extracted %d attention layers, %d heads, seq_len=%d",
        len(attention_layers),
        attention_layers[0].shape[0],
        len(tokens),
    )
    return tokens, attention_layers


tokens, attn_layers = extract_attention(DEFAULT_MODEL, DEFAULT_TEXT)
logger.info("Tokens: %s", tokens)

In [ ]:
import plotly.graph_objects as go


def plot_attention_heatmap(
    tokens: list[str],
    attention: torch.Tensor,
    layer: int = 0,
    head: int = 0,
    title: str = "Attention Heatmap",
) -> go.Figure:
    """Render a single (layer, head) attention matrix as a Plotly heatmap."""
    attn_matrix = attention[head].numpy()  # (seq, seq)

    fig = go.Figure(
        data=go.Heatmap(
            z=attn_matrix,
            x=tokens,
            y=tokens,
            colorscale="Blues",
            reversescale=False,
            showscale=True,
        )
    )
    fig.update_layout(
        title=f"{title} — Layer {layer}, Head {head}",
        xaxis_title="Key Tokens",
        yaxis_title="Query Tokens",
        yaxis=dict(autorange="reversed"),
        width=700,
        height=600,
    )
    return fig


fig = plot_attention_heatmap(tokens, attn_layers[0], layer=0, head=0, title=DEFAULT_MODEL)
fig.show()

if not WANDB_DISABLED:
    wandb.init(project="ai-attention-token-viz", config={"model": DEFAULT_MODEL})
    wandb.log({"attention_heatmap": wandb.Html(fig.to_html())})

In [ ]:
# Generate the Streamlit app script
streamlit_app = '''
"""Streamlit attention heatmap app.

Run with: streamlit run streamlit_app.py
"""
from __future__ import annotations

import os

import plotly.graph_objects as go
import streamlit as st
import torch
from transformers import AutoModel, AutoTokenizer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

st.set_page_config(page_title="Attention Heatmap Viz", layout="wide")
st.title("🔬 Transformer Attention Heatmap")
st.markdown(
    "Interactive visualisation of token-to-token attention weights.  \n"
    "Part of the [Transformer Research Hub](https://github.com/TylrDn/ai-transformer-research-hub)."
)

with st.sidebar:
    model_id = st.text_input("HuggingFace model", value="bert-base-uncased")
    text = st.text_area("Input text", value="The quick brown fox jumps over the lazy dog.")
    layer = st.slider("Layer", min_value=0, max_value=11, value=0)
    head = st.slider("Head", min_value=0, max_value=11, value=0)
    run_btn = st.button("Visualise")

if run_btn:
    with st.spinner("Loading model and extracting attention ..."):
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModel.from_pretrained(model_id, output_attentions=True)
        model = model.to(DEVICE).eval()
        enc = tokenizer(text, return_tensors="pt").to(DEVICE)
        tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
        with torch.no_grad():
            out = model(**enc)
        attn = out.attentions[layer][0, head].cpu().numpy()

    fig = go.Figure(data=go.Heatmap(z=attn, x=tokens, y=tokens, colorscale="Blues"))
    fig.update_layout(title=f"{model_id} — Layer {layer}, Head {head}",
                      yaxis=dict(autorange="reversed"), height=600)
    st.plotly_chart(fig, use_container_width=True)
'''

from pathlib import Path
Path("streamlit_app.py").write_text(streamlit_app.strip(), encoding="utf-8")
logger.info("Wrote streamlit_app.py — run with: streamlit run streamlit_app.py")